# Cosine Similarity Pipeline

This notebook combines two phases of the phishing-domain similarity analysis process:

1. **Part A - Computing Cosine Similarity** (`cosine_similarity.py`):  
   Loads PhishTank and Tranco embedding files and computes a cosine-similarity matrix, and saves the matrix to CSV.

2. **Part B - Extract Top-5 from a Pre-computed Matrix** (`top5_from_matrix.py`):  
   Streams the ~23GB of pre-existing matrix data (`FullMatrixOfCosineSimil.csv`) line-by-line and extract the top-5 most similar items per row **and** per column, keeping memory usage low.

---

## Imports

All libraries needed by both stages are loaded here.  

In [ ]:
# standard-library I/O
import csv
import json
import sys

# use min-heap for priority queue when choosing top-N matches
import heapq

# safe path handling across ipynb and Google resources
from pathlib import Path

# cosine similarity and other vector operations
import numpy as np

# used to easily export to CSV and handle dataframes
import pandas as pd

## Configuration

All our file paths and constants are defined in one place so they are easy to update.  
`BASE_DIR` is set to the directory containing this notebook for local running.
When embedding, I hit the limit of the 16GB of RAM on my machine, so the layout of the data is as follows:

embeds1 == 1st half of PhishTank Embeddings
embeds2 == 2nd half of PhishTank Embeddings
embedsA == All Tranco Embeddings

In [ ]:
BASE_DIR = Path.cwd()  # adjust if the notebook is not in the project root

# Part A I/O
EMBEDS1_FILE = BASE_DIR / "embeds1.csv"
EMBEDS2_FILE = BASE_DIR / "embeds2.csv"
EMBEDSA_FILE = BASE_DIR / "embedsA.csv"
OUT_MATRIX   = BASE_DIR / "cosine_similarity_matrix.csv" # Full matrix of cosine similarities

# Part B I/O
FULL_MATRIX_FILE = BASE_DIR / "FullMatrixOfCosineSimil.csv"
OUT_ROW          = BASE_DIR / "top5_per_row.csv"
OUT_COL          = BASE_DIR / "top5_per_column.csv"

TOP_N = 5 # Used if we wanted to test more or fewer top matches

---
# Part A - Compute Cosine Similarity from Raw Embeddings

As mentioned earlier, Phishtank took up too much memory. As such, we combine 2 sets of their embeddings
(`embeds1` + `embeds2`), deduplicate them, and then compute their cosine similarity against a third set (`embedsA`).  
The outputted result is a full N×M matrix plus a handy top-5 summary.

### Helper Function - `load_embeddings`

Reads a CSV whose rows are `(embedding_json_array, text_label)`.  
Returns a dict `{text_label: embedding_vector}`, keeping only the first  
occurrence of each label.

This deduplicates the data by keeping a sort of running tally for rows.

In [ ]:
def load_embeddings(filepath: Path) -> dict[str, list[float]]:
    """Load a CSV embeddings file (embedding,text), returning {text: embedding_vector}."""
    result = {}
    with open(filepath, "r", encoding="utf-8", newline="") as f:
        reader = csv.reader(f)
        next(reader, None)  # skip header row (embedding,text)
        for row in reader:
            if len(row) < 2:
                continue
            embedding_str, name = row[0], row[1]
            if name not in result:  # keep first occurrence
                result[name] = json.loads(embedding_str)
    return result


### Helper - `build_matrix`

Converts the `{name: vector}` dictionary into a NumPy float32 matrix  
and the corresponding ordered list of names.  
This is needed so we can use fast vectorised operations for Cosine Similarity.
This approach was ellucidated from the work of Saidani, cited in our paper [5].

In [ ]:
def build_matrix(app_dict: dict[str, list[float]]) -> tuple[list[str], np.ndarray]:
    """Convert {name: vector} dict to (name_list, numpy_matrix)."""
    names = list(app_dict.keys())
    mat = np.array([app_dict[n] for n in names], dtype=np.float32)
    return names, mat

### Helper - `cosine_similarity_matrix`

Computes the pairwise cosine similarity between two sets of vectors  
using the efficient **normalise → matmul** approach:

$$
\text{sim}(\mathbf{a}, \mathbf{b}) = \frac{\mathbf{a} \cdot \mathbf{b}}{\|\mathbf{a}\| \, \|\mathbf{b}\|}
$$

By L2-normalising each row first, the dot product of the resulting  
unit vectors directly yields cosine similarity. This basic utilization of Cosine Similarity is taken from Sonowal and Gunikhan [6].

In [ ]:
def cosine_similarity_matrix(A: np.ndarray, B: np.ndarray) -> np.ndarray:
    """
    Compute cosine similarity between every row in A and every row in B.
    Returns shape (len(A), len(B)).
    """
    A_norm = A / np.linalg.norm(A, axis=1, keepdims=True)
    B_norm = B / np.linalg.norm(B, axis=1, keepdims=True)
    return A_norm @ B_norm.T

### Step A1 - Load & Deduplicate Embeddings

Load `embeds1.csv` and `embeds2.csv`, merge them (with `embeds1` "winning"
on name collisions), and then load `embedsA.csv` separately.

In [ ]:
print("Loading embeddings files")
emb1 = load_embeddings(EMBEDS1_FILE)
emb2 = load_embeddings(EMBEDS2_FILE)

# Merge: emb1 takes precedence; add only new keys from emb2
combined = {}
for name, vec in emb1.items():
    combined[name] = vec
for name, vec in emb2.items():
    if name not in combined:
        combined[name] = vec
del emb1, emb2  # free memory

print(f"  Combined & deduplicated: {len(combined)} unique entries\n")

embA = load_embeddings(EMBEDSA_FILE)

### Step A2 - Build NumPy Arrays

Convert the Python dictionaries into dense float32 matrices  
to enable fast linear-algebra operations.

In [ ]:
print("Building numpy arrays")
names_12, mat_12 = build_matrix(combined)
del combined
names_A, mat_A = build_matrix(embA)
del embA

print(f"  Matrix 1&2: {mat_12.shape}  |  Matrix A: {mat_A.shape}")
mem_mb = (mat_12.nbytes + mat_A.nbytes) / 1e6
print(f"  Embedding memory: {mem_mb:.1f} MB")

### Step A3 - Compute Cosine Similarity

Multiply the two L2-normalised matrices to obtain the full  
N × M cosine-similarity matrix.

In [ ]:
print("Computing cosine similarity matrix")
sim = cosine_similarity_matrix(mat_12, mat_A)  # shape (N_12, N_A)
print(f"  Result shape: {sim.shape}  ({sim.shape[0] * sim.shape[1]:,} values)")

### Step A4 - Save Full Similarity Matrix

Write the entire matrix to `cosine_similarity_matrix.csv`.  
Row index = items from Embeds 1 & 2; columns = items from Embeds A.

In [ ]:
print("Saving full similarity matrix")
df_full = pd.DataFrame(sim, index=names_12, columns=names_A)
df_full.index.name = "App_Embeds12"
df_full.to_csv(OUT_MATRIX, float_format="%.6f")
print(f"  Saved: {OUT_MATRIX}  ({OUT_MATRIX.stat().st_size / 1e6:.1f} MB)")

---
# Part B - Stream-Extract Top-5 from a Large Pre-computed Matrix

When the full similarity matrix already exists on disk  
(`FullMatrixOfCosineSimil.csv` at 20+ GB),  
we can't load it all into RAM.

Instead, this section **streams the file row-by-row** and maintains  
fixed-size min-heaps (size 5) to track the best matches:

| Output file | Answers the question |
|---|---|
| `top5_per_row.csv` | For each **row** item, which 5 column items are most similar? |
| `top5_per_column.csv` | For each **column** item, which 5 row items are most similar? |

Memory stays bounded: only the header row plus one heap (≤ 5 entries)  
per column are held in RAM at any time.

### Step B1 - Read the Matrix Header

Open the CSV just to read the first line.  
This gives us the column labels and lets us allocate one min-heap per column.

In [ ]:
print("Reading header …", flush=True)
with open(FULL_MATRIX_FILE, "r", encoding="utf-8", newline="") as f:
    reader = csv.reader(f)
    header = next(reader)

# header[0] is the corner label (e.g. "App_Embeds12"), rest are column names
col_labels = header[1:]
num_cols = len(col_labels)
print(f"  {num_cols} columns detected.")

### Step B2 - Stream Rows & Collect Top-5 per Row and Column

For **each row** we maintain a min-heap of size `TOP_N` tracking  
the highest-similarity column items seen so far.  
Simultaneously, for **each column** we maintain a parallel min-heap  
tracking the highest-similarity row items.

Because a min-heap keeps the *smallest* element at the top,  
we can efficiently evict it whenever a larger value arrives,  
guaranteeing O(R · C · log N) total work with O(C · N) memory according to Datacamp [9].

In [ ]:
# Min-heaps for each column: heap of (similarity, row_label)
col_heaps: list[list[tuple[float, str]]] = [[] for _ in range(num_cols)]

print("Streaming rows …", flush=True)
row_results: list[tuple[str, list[tuple[float, str]]]] = []
row_count = 0

with open(FULL_MATRIX_FILE, "r", encoding="utf-8", newline="") as f:
    reader = csv.reader(f)
    next(reader)  # skip header

    for row in reader:
        row_label = row[0]
        values = row[1:]

        # --- top 5 for this row (min-heap of size TOP_N) ---
        row_pairs: list[tuple[float, str]] = []
        for j, val_str in enumerate(values):
            if j >= num_cols:
                break
            try:
                sim_val = float(val_str)
            except (ValueError, IndexError):
                continue

            # Row top-5
            if len(row_pairs) < TOP_N:
                heapq.heappush(row_pairs, (sim_val, col_labels[j]))
            elif sim_val > row_pairs[0][0]:
                heapq.heapreplace(row_pairs, (sim_val, col_labels[j]))

            # Column top-5
            heap = col_heaps[j]
            if len(heap) < TOP_N:
                heapq.heappush(heap, (sim_val, row_label))
            elif sim_val > heap[0][0]:
                heapq.heapreplace(heap, (sim_val, row_label))

        # Store row result sorted descending
        row_pairs.sort(reverse=True)
        row_results.append((row_label, row_pairs))

        row_count += 1
        if row_count % 5000 == 0:
            print(f"  {row_count:,} rows processed", flush=True)

print(f"  Done: {row_count:,} rows")

### Step B3 - Write Top-5 per Row

Each output row contains the item label, the names of its top-5 matches,  
and the corresponding similarity scores.

In [ ]:
print(f"Writing {OUT_ROW.name} …", flush=True)
with open(OUT_ROW, "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(
        ["item"]
        + [f"match_{i+1}" for i in range(TOP_N)]
        + [f"score_{i+1}" for i in range(TOP_N)]
    )
    for row_label, pairs in row_results:
        names = [p[1] for p in pairs]
        scores = [f"{p[0]:.6f}" for p in pairs]
        while len(names) < TOP_N:
            names.append("")
            scores.append("")
        writer.writerow([row_label] + names + scores)

print(f"  Saved: {OUT_ROW}")

### Step B4 - Write Top-5 per Column

Same format, but now each row of the output represents one  
**column** item from the original matrix, along with the 5 row  
items that were most similar to it.

In [ ]:
print(f"Writing {OUT_COL.name} …", flush=True)
with open(OUT_COL, "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(
        ["item"]
        + [f"match_{i+1}" for i in range(TOP_N)]
        + [f"score_{i+1}" for i in range(TOP_N)]
    )
    for j, col_label in enumerate(col_labels):
        pairs = sorted(col_heaps[j], reverse=True)
        names = [p[1] for p in pairs]
        scores = [f"{p[0]:.6f}" for p in pairs]
        while len(names) < TOP_N:
            names.append("")
            scores.append("")
        writer.writerow([col_label] + names + scores)

print(f"  Saved: {OUT_COL}")

### Part B Summary

In [ ]:
print("Part B complete")
print(f"  Rows streamed:  {row_count:,}")
print(f"  Columns:        {num_cols:,}")
print(f"\nOutputs:")
print(f"  → {OUT_ROW}")
print(f"  → {OUT_COL}")

You can see all of the larger embedded data at the below URLs:

[https://storage.googleapis.com/gscsteam1-colabresultbucket/](https://storage.googleapis.com/gscsteam1-colabresultbucket/)

[https://drive.google.com/drive/folders/1vULwJLjEuPXAWCR7eL5kq6g5KK3VU2wf?usp=sharing](https://drive.google.com/drive/folders/1vULwJLjEuPXAWCR7eL5kq6g5KK3VU2wf?usp=sharing)

## Citations
[1] Gabriela Brezeanu, Alexandru Archip, and Codrut-Georgian Artene. Phish fighter: Self updating machine learning shield against phishing kits based on HTML code analysis. 13:4460–4486.

[2] Bibhu Dash and Meraj Farheen Ansari. An effective cybersecurity awareness training model: First defense of an organizational security strategy. International Research Journal of Engineering and Technology, 9:2395–0056, 04 2022.

[3] Dawn M. Sarno, Maggie W. Harris, and Jeffrey Black. Which phish is captured in the net? understanding phishing susceptibility and individual differences. 37(4):789–803._eprint:https://onlinelibrary.wiley.com/doi/pdf/10.1002/acp.4075.

[4] Giuseppe Desolda, Francesco Greco, and Luca Vigano. APOLLO: A GPT-based tool to detect phishing emails and generate explanations that warn users. 9(4):EICS003:1–EICS003:33.

[5] Nadjate Saidani, Kamel Adi, and Mohand Saïd Allili. A semantic-based classification approach for an enhanced spam detection. Computers & Security, 94, 2020-07.

[6] Sonowal and Gunikhan. Detecting phishing sms based on multiple correlation algorithms. SN Computer Science, 1, 11 2020.

[7] Panpan Zhang, Jing Ya, Tingwen Liu, Quangang Li, Jinqiao Shi, and Zhaojun Gu. imcircle: Automatic mining of indicators of compromise from the web. In 2019 IEEE  symposium on Computers and Communications (ISCC), 2019.

[8] Rasha Zieni, Luisa Massari, and Maria Carla Calzarossa. Phishing or not phishing? a survey on the detection of phishing websites. 11:18499–18519.

[9] Chugani, Vinod. “What Is Cosine Distance?” Datacamp.com, DataCamp, 28 July 2024, www.datacamp.com/tutorial/cosine-distance.

[10] Cisco Systems, Inc. PhishTank: Join the fight against phishing. phishtank.com, 2026.

[11] Victor Le Pochat, Tom Van Goethem, Samaneh Tajalizadehkhoob, Maciej Korczy´nski, and Wouter Joosen. Tranco: A Research-Oriented Top Sites Ranking Hardened Against Manipulation. In Proceedings of the 26th Annual Network and Distributed System Security Symposium (NDSS), 2019.